In [ ]:
import os
from datasets import load_dataset
from matplotlib import pyplot as plt
import pandas as pd
import open_clip

from langchain_experimental.open_clip import OpenCLIPEmbeddings
from langchain+teddynote.models import MultiModal
from langchain_openai import ChatOpenAI

In [ ]:
# COCO 데이터셋

dataset = load_dataset(path="detection-datasets/coco", name="default", split="train", streaming=True)

In [ ]:
 이미지 저장 폴더와 이미지 개수 설정
IMAGE_FOLDER = "tmp"
N_IMAGES = 20

# 그래프 플로팅을 위한 설정
plot_cols = 5
plot_rows = N_IMAGES // plot_cols
fig, axes = plt.subplots(plot_rows, plot_cols, figsize=(plot_rows * 2, plot_cols * 2))
axes = axes.flatten()

# 이미지를 폴더에 저장하고 그래프에 표시
dataset_iter = iter(dataset)
os.makedirs(IMAGE_FOLDER, exist_ok=True)
for i in range(N_IMAGES):
    # 데이터셋에서 이미지와 레이블 추출
    data = next(dataset_iter)
    image = data["image"]
    label = data["objects"]["category"][0]  # 첫 번째 객체의 카테고리를 레이블로 사용

    # 그래프에 이미지 표시 및 레이블 추가
    axes[i].imshow(image)
    axes[i].set_title(label, fontsize=8)
    axes[i].axis("off")

    # 이미지 파일로 저장
    image.save(f"{IMAGE_FOLDER}/{i}.jpg")

# 그래프 레이아웃 조정 및 표시
plt.tight_layout()
plt.show()

멀티모달 임베딩

In [ ]:
pd.DataFrame(open_clip.list_pretrained(), columns=["model_name", "checkpoint"]).head(10)

In [ ]:
image_embedding_function = OpenCLIPEmbeddings(model_name="ViT-H-14-378-quickgelu", checkpoint="dfn5b")

In [ ]:
# 이미지의 경로를 리스트로 저장
image_uris = sorted([os.path.join("tmp", image_name) for image_name in os.listdir("tmp") if image_name.endswith(".jpg")])

image_uris

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")

model = MultiModal(
    model=llm, 
    system_prompt="Your mission is to describe the image in detail", 
    user_prompt="Description should be written in one sentence(less than 60 characters)"
)

In [ ]:
model.invoke(image_urls[0])  # 이미지에 대한 설명 생성